# Hand Gesture Recognition from MediaPipe Landmarks

This notebook is a Colab friendly version of the training script.
It reads hand landmark data, builds a small dataset, trains a few models
to guess the gesture, and shows charts and numbers about how well each
model did.

Gestures it learns to tell apart: call, dislike, like, take_picture

To use the T4 GPU in Colab go to Runtime, then Change runtime type,
then pick T4 GPU as the hardware accelerator, then save.

The Random Forest, SVM, and MLP from scikit learn only run on CPU, since
that library does not support GPU. To actually put the T4 GPU to work,
this notebook also trains a small PyTorch neural network, which will
automatically run on the GPU if one is available.

In [ ]:
# check if a GPU is available and print what Colab gave us
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. Go to Runtime, then Change runtime type, then pick T4 GPU.")

## Step 1: Get the project data into Colab

Upload your `project data` folder (the one containing `landmarks.json`)
to this Colab session, or mount your Google Drive and point `DATA_DIR`
at it. Run only one of the two cells below.

In [ ]:
# option A: upload files directly from your computer
# this opens a file picker, use it to upload landmarks.json
# (skip this cell if you are using Google Drive instead)
from google.colab import files
uploaded = files.upload()

In [ ]:
# option B: mount Google Drive and point at the folder there
# (skip this cell if you already uploaded files directly above)
from google.colab import drive
drive.mount("/content/drive")
# change this path to wherever your project data folder actually is
DATA_DIR = "/content/drive/MyDrive/project data"

In [ ]:
# if you used option A (direct upload) instead of Drive, use this path
# comment this line out if you used option B above
DATA_DIR = "."
OUT_DIR = "./outputs"


## Step 2: Install and import everything we need

In [ ]:
# these libraries come preinstalled on Colab, this just makes sure
!pip install -q scikit-learn matplotlib numpy

In [ ]:
# json lets us read and write .json files
import json
# os lets us build file paths and make folders
import os
# numpy helps us do math on lists of numbers quickly
import numpy as np
# matplotlib is the library we use to draw charts
import matplotlib.pyplot as plt

# this splits our data into a training group and a testing group
from sklearn.model_selection import train_test_split
# this rescales numbers so they are easier for models to learn from
from sklearn.preprocessing import StandardScaler
# this is one type of model we will train, called a Random Forest
from sklearn.ensemble import RandomForestClassifier
# this is another type of model, called a Support Vector Machine
from sklearn.svm import SVC
# this is a third type of model, a small neural network on CPU
from sklearn.neural_network import MLPClassifier
# these are tools that measure how good our models are
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_fscore_support
)

# torch tools for the GPU trained neural network
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# a fixed number so every run of this program splits and trains the same way
RANDOM_STATE = 42
# the four gesture names we are trying to recognize, in a fixed order
CLASSES = ["call", "dislike", "like", "take_picture"]

# make the output folder if it does not already exist
os.makedirs(OUT_DIR, exist_ok=True)

## Step 3: Load the landmark data and turn it into numbers

In [ ]:
# this function opens the landmarks file and gives back its contents
def load_landmarks(json_path):
    # open the file for reading
    with open(json_path, "r") as f:
        # turn the file's text into a Python list or dictionary
        data = json.load(f)
    # send that data back to whoever called this function
    return data

In [ ]:
# this function turns one image's hand data into a simple list of numbers
def featurize(entry):
    """
    Turn one MediaPipe hand_landmarks record into a fixed size list of
    numbers that a model can learn from.

    What it does, step by step:
      1. Look at the first hand found in the picture.
      2. Move all points so the wrist point becomes zero, meaning 0, 0, 0.
      3. Shrink or grow all points using the size of the hand, so a hand
         close to the camera and a hand far from the camera look the same.
      4. Turn the 21 points, each with x, y, z, into one long list of 63 numbers.

    If no hand was found in the picture, this returns None instead, so we
    can skip that picture later instead of guessing wrong numbers for it.
    """
    # get the list of hands found in this picture, if any
    lm_list = entry.get("hand_landmarks")
    # if there is no hand list, or it is empty, we have nothing to use
    if not lm_list or len(lm_list) == 0:
        # tell the caller this picture has no usable hand
        return None
    # take just the first hand found, some pictures could have more than one
    pts = lm_list[0]
    # if that first hand somehow has no points, skip it too
    if not pts:
        return None

    # build a table of numbers, one row per point, three columns for x, y, z
    coords = np.array([[p["x"], p["y"], p["z"]] for p in pts], dtype=np.float64)
    # remember where the wrist point is, point number 0
    wrist = coords[0].copy()
    # move every point so the wrist becomes the new zero point
    coords -= wrist  # this makes the hand's position in the picture not matter

    # measure the distance from the wrist to the middle finger's base knuckle
    scale = np.linalg.norm(coords[9])  # this tells us roughly how big the hand looks
    # avoid dividing by a number that is basically zero
    if scale < 1e-8:
        scale = 1e-8
    # shrink or grow all points using that distance, so hand size does not matter
    coords /= scale  # this makes the hand's size or distance from camera not matter

    # turn the table of 21 rows and 3 columns into one flat list of 63 numbers
    return coords.flatten()

In [ ]:
# this function builds the full list of feature rows and their matching labels
def build_dataset(records):
    # X will hold the feature numbers, y will hold the matching gesture name
    X, y, dropped = [], [], 0
    # go through every picture's record one at a time
    for r in records:
        # turn this record into a list of numbers, or None if no hand was found
        feat = featurize(r)
        # if there was no usable hand, count it and skip to the next picture
        if feat is None:
            dropped += 1
            continue
        # otherwise save the numbers
        X.append(feat)
        # and save the correct gesture name for those numbers
        y.append(r["label"])
    # turn the plain lists into numpy arrays and also return how many we dropped
    return np.array(X), np.array(y), dropped

In [ ]:
# load all the landmark records from the json file
records = load_landmarks(os.path.join(DATA_DIR, "landmarks.json"))
# print how many records we loaded, just so we can see progress
print(f"Loaded {len(records)} landmark records.")

## Step 4: Look at the data before training anything

In [ ]:
# this function draws a bar chart showing how many pictures are in each class
def plot_class_distribution(records, out_dir):
    # count how many pictures belong to each gesture
    counts = {c: sum(1 for r in records if r["label"] == c) for c in CLASSES}
    # start a new blank chart of a certain size
    plt.figure(figsize=(6, 4))
    # draw one bar per gesture, using the counts we just made
    plt.bar(counts.keys(), counts.values(), color="#4C72B0")
    # give the chart a title
    plt.title("Class distribution of raw dataset")
    # label the up and down axis
    plt.ylabel("Number of samples")
    # tidy up the spacing so labels are not cut off
    plt.tight_layout()
    # save the chart as an image file in the output folder
    plt.savefig(os.path.join(out_dir, "class_distribution.png"), dpi=150)
    # show the chart right here in the notebook
    plt.show()

plot_class_distribution(records, OUT_DIR)

In [ ]:
# this function draws a chart of how often MediaPipe failed to find a hand
def plot_missing_detection(records, out_dir):
    # start a counter of missed pictures for each gesture, starting at zero
    miss = {c: 0 for c in CLASSES}
    # start a counter of total pictures for each gesture, starting at zero
    total = {c: 0 for c in CLASSES}
    # look at every picture's record
    for r in records:
        # add one to the total count for this gesture
        total[r["label"]] += 1
        # if this record has no hand landmarks at all
        if not r.get("hand_landmarks"):
            # add one to the missed count for this gesture
            miss[r["label"]] += 1
    # turn the raw counts into percentages for each gesture
    rates = {c: 100.0 * miss[c] / total[c] for c in CLASSES}
    # start a new blank chart
    plt.figure(figsize=(6, 4))
    # draw one bar per gesture showing its percentage of missed detections
    plt.bar(rates.keys(), rates.values(), color="#C44E52")
    # give the chart a title
    plt.title("MediaPipe hand detection failure rate by class")
    # label the up and down axis
    plt.ylabel("Percent of images with no hand detected")
    # tidy up spacing
    plt.tight_layout()
    # save the chart to a file
    plt.savefig(os.path.join(out_dir, "missing_detection_rate.png"), dpi=150)
    # show the chart right here in the notebook
    plt.show()
    # send back the raw counts in case the caller wants them too
    return miss, total

miss, total = plot_missing_detection(records, OUT_DIR)
print("Missing hand detection counts per class:", miss)

In [ ]:
# turn all the records into a table of numbers, X, and matching labels, y
X, y, dropped = build_dataset(records)
# print the shape of our data and how many pictures we had to skip
print(f"Built feature matrix: X={X.shape}, dropped={dropped} samples with no hand detected")

## Step 5: Split the data and scale it

In [ ]:
# split the data into 70 percent for training and 30 percent for later use
# stratify keeps the same class balance in both pieces
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
# split that remaining 30 percent in half, 15 percent validation, 15 percent test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)
# print how many pictures ended up in each of the three groups
print(f"Split sizes: train {len(X_train)}, val {len(X_val)}, test {len(X_test)}")

# create a tool that will rescale our numbers to a standard range
scaler = StandardScaler()
# learn the rescaling from the training data, and apply it to the training data
X_train_s = scaler.fit_transform(X_train)
# apply that same rescaling to the validation data, without relearning it
X_val_s = scaler.transform(X_val)
# apply that same rescaling to the test data too
X_test_s = scaler.transform(X_test)

## Step 6: Train the CPU models

Random Forest, SVM, and the scikit learn MLP all run on CPU only,
since scikit learn has no GPU support. They are fast enough on CPU
because our feature vectors are tiny, only 63 numbers each.

In [ ]:
# this function draws a confusion matrix, which shows correct versus wrong guesses
def plot_confusion(cm, labels, out_dir, model_name):
    # start a new blank chart of a certain size
    plt.figure(figsize=(5.5, 5))
    # show the confusion matrix as a grid of colored squares
    plt.imshow(cm, cmap="Blues")
    # give the chart a title that includes the model's name
    plt.title(f"Confusion matrix for {model_name}")
    # add a color scale bar on the side
    plt.colorbar()
    # make a list of positions for the tick marks, one per gesture
    tick = np.arange(len(labels))
    # put gesture names along the bottom, tilted so they fit
    plt.xticks(tick, labels, rotation=45, ha="right")
    # put gesture names along the side
    plt.yticks(tick, labels)
    # go through every row of the grid
    for i in range(len(labels)):
        # go through every column of the grid
        for j in range(len(labels)):
            # write the actual number in that square, using white or black text
            # depending on how dark the square's background color is
            plt.text(j, i, cm[i, j], ha="center", va="center",
                      color="white" if cm[i, j] > cm.max() / 2 else "black")
    # label the side axis
    plt.ylabel("True label")
    # label the bottom axis
    plt.xlabel("Predicted label")
    # tidy up spacing
    plt.tight_layout()
    # pick a plain file name for the Random Forest model, or a custom one for others
    fname = "confusion_matrix.png" if model_name == "Random Forest" else f"confusion_matrix_{model_name.replace(' ', '_')}.png"
    # save the chart to a file
    plt.savefig(os.path.join(out_dir, fname), dpi=150)
    # show the chart right here in the notebook
    plt.show()

In [ ]:
# set up the two scikit learn models we want to try, with their settings
cpu_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "SVM (RBF)": SVC(kernel="rbf", C=10, gamma="scale", random_state=RANDOM_STATE),
    "MLP (CPU)": MLPClassifier(
        hidden_layer_sizes=(64, 32), max_iter=1000, random_state=RANDOM_STATE
    ),
}

# this will store the results for every model we try
results = {}
# go through each CPU model one at a time
for name, model in cpu_models.items():
    # teach the model using the training data
    model.fit(X_train_s, y_train)
    # check how accurate it is on the validation data
    val_acc = accuracy_score(y_val, model.predict(X_val_s))
    # use the model to guess labels for the test data
    y_pred_test = model.predict(X_test_s)
    # check how accurate those guesses were compared to the real labels
    test_acc = accuracy_score(y_test, y_pred_test)
    # save both accuracy numbers for this model
    results[name] = {"val_accuracy": val_acc, "accuracy": test_acc, "model": model}
    # print the accuracy numbers so we can see progress as it runs
    print(f"{name}: val_acc={val_acc:.4f}, test_acc={test_acc:.4f}")
    # build a confusion matrix comparing real labels to guessed labels
    cm = confusion_matrix(y_test, y_pred_test, labels=CLASSES)
    # draw and save a chart of that confusion matrix
    plot_confusion(cm, CLASSES, OUT_DIR, name)

## Step 7: Train a neural network on the T4 GPU

This part uses PyTorch instead of scikit learn, so it will actually
run on the T4 GPU if one is turned on. The model is small, since our
input is only 63 numbers, but this shows the proper GPU training setup
you would scale up for a bigger model or bigger dataset.

In [ ]:
# turn the gesture names into numbers, since PyTorch needs numeric labels
label_to_idx = {c: i for i, c in enumerate(CLASSES)}
y_train_idx = np.array([label_to_idx[label] for label in y_train])
y_val_idx = np.array([label_to_idx[label] for label in y_val])
y_test_idx = np.array([label_to_idx[label] for label in y_test])

# turn our numpy arrays into PyTorch tensors, using float32 for the features
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_idx, dtype=torch.long)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
y_val_t = torch.tensor(y_val_idx, dtype=torch.long)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test_idx, dtype=torch.long)

# wrap the training tensors so we can loop over them in small batches
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
# define a small neural network with two hidden layers
class GestureNet(nn.Module):
    def __init__(self, input_size=63, num_classes=4):
        # set up the parent class first
        super().__init__()
        # first hidden layer, shrinks 63 numbers down to 64
        self.fc1 = nn.Linear(input_size, 64)
        # second hidden layer, shrinks 64 numbers down to 32
        self.fc2 = nn.Linear(64, 32)
        # output layer, turns 32 numbers into one score per gesture class
        self.fc3 = nn.Linear(32, num_classes)
        # relu adds non linearity between layers so the network can learn curves
        self.relu = nn.ReLU()
        # dropout randomly turns off some connections during training to avoid overfitting
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # pass through the first layer then relu
        x = self.relu(self.fc1(x))
        # randomly drop some values
        x = self.dropout(x)
        # pass through the second layer then relu
        x = self.relu(self.fc2(x))
        # pass through the final layer to get class scores
        x = self.fc3(x)
        return x

# create the model and move it onto the GPU if one is available
gesture_net = GestureNet(input_size=X_train_s.shape[1], num_classes=len(CLASSES)).to(device)
print(gesture_net)

In [ ]:
# cross entropy loss is the standard choice for multi class classification
criterion = nn.CrossEntropyLoss()
# adam is a reliable, commonly used optimizer for neural networks
optimizer = optim.Adam(gesture_net.parameters(), lr=0.001)

# move the validation and test tensors onto the same device as the model
X_val_gpu = X_val_t.to(device)
y_val_gpu = y_val_t.to(device)
X_test_gpu = X_test_t.to(device)
y_test_gpu = y_test_t.to(device)

# how many times we loop over the whole training set
num_epochs = 60
# keep track of accuracy after each epoch so we can plot it later
train_acc_history = []
val_acc_history = []

# loop over the dataset this many times
for epoch in range(num_epochs):
    # put the model in training mode, which turns dropout on
    gesture_net.train()
    correct = 0
    total = 0
    # loop over each small batch of training data
    for batch_x, batch_y in train_loader:
        # move this batch onto the GPU
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        # clear out gradients from the previous step
        optimizer.zero_grad()
        # run the batch through the model to get predictions
        outputs = gesture_net(batch_x)
        # measure how wrong those predictions were
        loss = criterion(outputs, batch_y)
        # figure out how to adjust the weights to reduce that error
        loss.backward()
        # actually update the weights
        optimizer.step()
        # count how many predictions in this batch were correct
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    # calculate training accuracy for this epoch
    train_acc = correct / total
    train_acc_history.append(train_acc)

    # put the model in evaluation mode, which turns dropout off
    gesture_net.eval()
    # we do not need gradients when just checking accuracy
    with torch.no_grad():
        val_outputs = gesture_net(X_val_gpu)
        _, val_predicted = torch.max(val_outputs, 1)
        val_acc = (val_predicted == y_val_gpu).float().mean().item()
    val_acc_history.append(val_acc)

    # print progress every 10 epochs so the log is not too long
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1}/{num_epochs}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

In [ ]:
# plot how training and validation accuracy changed over time
plt.figure(figsize=(6, 4))
plt.plot(train_acc_history, label="Train accuracy")
plt.plot(val_acc_history, label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("GPU neural network training curve")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "gpu_training_curve.png"), dpi=150)
plt.show()

In [ ]:
# switch to evaluation mode for the final test check
gesture_net.eval()
# turn off gradient tracking since we are only measuring, not training
with torch.no_grad():
    test_outputs = gesture_net(X_test_gpu)
    _, test_predicted = torch.max(test_outputs, 1)
    gpu_test_acc = (test_predicted == y_test_gpu).float().mean().item()

print(f"PyTorch GPU model test accuracy: {gpu_test_acc:.4f}")

# turn the predicted numbers back into gesture names for the confusion matrix
y_pred_gpu_labels = [CLASSES[i] for i in test_predicted.cpu().numpy()]
cm_gpu = confusion_matrix(y_test, y_pred_gpu_labels, labels=CLASSES)
plot_confusion(cm_gpu, CLASSES, OUT_DIR, "PyTorch GPU MLP")

# add this model's result to the same results dictionary as the others
results["PyTorch MLP (GPU)"] = {"val_accuracy": val_acc_history[-1], "accuracy": gpu_test_acc}

## Step 8: Compare every model side by side

In [ ]:
# this function draws a bar chart comparing the accuracy of each model
def plot_model_comparison(results, out_dir):
    # get the list of model names
    names = list(results.keys())
    # get the matching accuracy score for each model
    accs = [results[n]["accuracy"] for n in names]
    # start a new blank chart
    plt.figure(figsize=(7, 4))
    # draw one bar per model, using its accuracy as the bar height
    bars = plt.bar(names, accs, color=["#55A868", "#4C72B0", "#C44E52", "#8172B2"])
    # fix the up and down axis to go from 0 to 1, since accuracy is a fraction
    plt.ylim(0, 1.0)
    # label the up and down axis
    plt.ylabel("Test accuracy")
    # give the chart a title
    plt.title("Model comparison on held out test set")
    # tilt the model names so long labels are readable
    plt.xticks(rotation=20, ha="right")
    # write the exact accuracy number above each bar
    for b, a in zip(bars, accs):
        plt.text(b.get_x() + b.get_width() / 2, a + 0.01, f"{a:.3f}", ha="center")
    # tidy up spacing
    plt.tight_layout()
    # save the chart to a file
    plt.savefig(os.path.join(out_dir, "model_comparison.png"), dpi=150)
    # show the chart right here in the notebook
    plt.show()

plot_model_comparison(results, OUT_DIR)

# figure out which model had the best test accuracy overall
best_name = max(results, key=lambda n: results[n]["accuracy"])
best_acc = results[best_name]["accuracy"]
print(f"Best model: {best_name} with test accuracy {best_acc:.4f}")

In [ ]:
# this function draws a chart comparing precision, recall, and F1 for each class
def plot_per_class_metrics(y_test, y_pred, labels, out_dir, title_suffix):
    # calculate precision, recall, and F1 score for every gesture class
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, labels=labels, zero_division=0)
    # make a list of positions, one per gesture, to place bars at
    x = np.arange(len(labels))
    # decide how wide each small bar should be
    width = 0.25
    # start a new blank chart
    plt.figure(figsize=(7, 4))
    # draw the precision bars, shifted slightly left
    plt.bar(x - width, p, width, label="Precision")
    # draw the recall bars, in the middle
    plt.bar(x, r, width, label="Recall")
    # draw the F1 bars, shifted slightly right
    plt.bar(x + width, f1, width, label="F1")
    # label each group of bars with its gesture name
    plt.xticks(x, labels)
    # fix the up and down axis so it is easy to compare charts
    plt.ylim(0, 1.05)
    # give the chart a title
    plt.title(f"Per class precision, recall, and F1 for {title_suffix}")
    # show a small legend explaining the bar colors
    plt.legend()
    # tidy up spacing
    plt.tight_layout()
    # save the chart to a file
    plt.savefig(os.path.join(out_dir, "per_class_metrics.png"), dpi=150)
    # show the chart right here in the notebook
    plt.show()
    # send back the three lists of numbers in case they are needed later
    return p, r, f1

# use whichever model scored best, and get its predictions on the test set again
if best_name == "PyTorch MLP (GPU)":
    y_pred_best = y_pred_gpu_labels
else:
    y_pred_best = results[best_name]["model"].predict(X_test_s)

# build a detailed report of precision, recall, and F1 for the best model
report_dict = classification_report(y_test, y_pred_best, labels=CLASSES, output_dict=True, zero_division=0)
p, r, f1 = plot_per_class_metrics(y_test, y_pred_best, CLASSES, OUT_DIR, best_name)
print(json.dumps(report_dict, indent=2))

## Step 9: Save a summary file with all the results

In [ ]:
# gather everything worth remembering about this run into one dictionary
summary = {
    "n_total_records": len(records),
    "n_dropped_no_hand": int(dropped),
    "missing_detection_by_class": miss,
    "split_sizes": {"train": len(X_train), "val": len(X_val), "test": len(X_test)},
    "model_results": {n: {"val_accuracy": v["val_accuracy"], "accuracy": v["accuracy"]} for n, v in results.items()},
    "best_model": best_name,
    "best_test_accuracy": best_acc,
    "classification_report_best_model": report_dict,
    "device_used_for_gpu_model": str(device),
}
# open a new file for writing and save the summary as json text
with open(os.path.join(OUT_DIR, "metrics_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("Saved metrics_summary.json")
print(json.dumps({k: v for k, v in summary.items() if k != "classification_report_best_model"}, indent=2))

## Step 10: Download your results

If you are running this in Colab and want to bring the charts and
summary back to your own computer, run the cell below to zip and
download the output folder.

In [ ]:
# zip up the whole output folder so it is easy to download in one file
import shutil
shutil.make_archive("gesture_results", "zip", OUT_DIR)

from google.colab import files
files.download("gesture_results.zip")